# CNU Q&A 챗봇 — Colab 실행 노트북

**런타임 설정: T4 GPU 필수** (런타임 > 런타임 유형 변경 > T4 GPU)

---

## 처음 실행 시 순서
```
셀 1 실행 → [런타임 다시 시작] → 셀 2 → 3 → 4 → 5 → 6 → 7 → 8
```

## 이후 실행 시 (세션 재시작 후)
```
셀 1 → [런타임 다시 시작] → 셀 2 → 3 → 4 확인 → 7 → 8 (또는 9 / 10)
```
> 셀 5 · 6은 `chunks.json` 변경 시에만 재실행

---

## 사용 모델
| 구분 | 모델 | 비고 |
|------|------|------|
| 임베딩 | nlpai-lab/KURE-v1 | 1024d, 한국어 retrieval 특화 |
| LLM | Qwen/Qwen2.5-3B-Instruct | 4-bit NF4, ~2.5GB VRAM |
| 검색 | BM25(Kiwi) + 임베딩 RRF | k=60 하이브리드 |

In [ ]:
# [1] 패키지 설치 — 처음 1회만 실행
# 완료 후: 런타임 > 런타임 다시 시작 → 셀 2부터 실행
!pip install -q 'chromadb==0.5.23' --force-reinstall
!pip install -q 'sentence-transformers>=2.6.0' 'kiwipiepy>=0.17.0' 'rank-bm25>=0.2.2'
!pip install -q 'transformers>=4.40.0' 'peft>=0.10.0' 'bitsandbytes>=0.43.0' 'accelerate>=0.28.0'
!pip install -q 'gradio>=4.20.0'
print('설치 완료 — 런타임 > 런타임 다시 시작 후 셀 2부터 실행하세요')

In [ ]:
# [2] Google Drive 마운트 & 프로젝트 경로 설정
from google.colab import drive
drive.mount('/content/drive')

import sys, os

# 폴더명이 다르면 이 줄만 수정 (확인: !ls /content/drive/MyDrive/)
PROJECT = '/content/drive/MyDrive/CNU-QA-chatbot'

if not os.path.exists(PROJECT):
    print('폴더 없음: ' + PROJECT)
    print('!ls /content/drive/MyDrive/ 로 실제 폴더명 확인 후 PROJECT 수정')
else:
    sys.path.insert(0, PROJECT)  # from src.xxx import ... 가능
    os.chdir(PROJECT)             # 상대 경로 기준점 = 프로젝트 루트
    print('작업 디렉토리: ' + os.getcwd())
    print('sys.path[0] : ' + sys.path[0])

In [ ]:
# [3] GPU 확인
import torch
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram  = props.total_memory / 1e9
    print('GPU  : ' + props.name)
    print('VRAM : ' + str(round(vram, 1)) + ' GB')
    print('필요 : KURE-v1 ~1.2GB + Qwen2.5-3B 4-bit ~2.5GB = 합계 ~3.7GB')
else:
    print('GPU 없음 — 런타임 > 런타임 유형 변경 > T4 GPU 선택 후 재시작')

In [ ]:
# [4] 데이터 확인
import json, os
chunks_path = os.path.join(PROJECT, 'data', 'processed', 'chunks.json')
db_path     = os.path.join(PROJECT, 'chroma_db')

if os.path.exists(chunks_path):
    with open(chunks_path, encoding='utf-8') as f:
        chunks = json.load(f)
    print('chunks.json : ' + str(len(chunks)) + '청크')
    print('샘플        : [' + chunks[0]['category'] + '] ' + chunks[0]['title'][:40])
else:
    print('chunks.json 없음')
    print('로컬에서 python src/preprocessing/preprocess.py 실행 후 Drive에 업로드')

if os.path.exists(db_path):
    print('chroma_db   : 존재 -> 셀 7로 바로 이동 가능')
else:
    print('chroma_db   : 없음  -> 셀 5 -> 셀 6 실행 필요')

In [ ]:
# [5] chroma_db 초기화 — 처음 실행 또는 chunks.json 변경 시만 실행
# chroma_db 존재하고 데이터 변경 없으면 이 셀 건너뛰기
import shutil, os
db_path = os.path.join(PROJECT, 'chroma_db')
if os.path.exists(db_path):
    shutil.rmtree(db_path)
    print('chroma_db 삭제 완료 -> 셀 6 실행')
else:
    print('chroma_db 없음 (정상) -> 셀 6 실행')

In [ ]:
# [6] 벡터 DB 구축 — 처음 또는 셀 5 실행 후에만 필요
# KURE-v1로 모든 청크를 임베딩해 Drive의 chroma_db/ 에 영구 저장
# 소요 시간: 약 5~10분 (청크 수 및 GPU 성능에 따라 다름)
from src.vectordb.build_db import build
build()
print('벡터 DB 구축 완료 -> 셀 7로 이동')

In [ ]:
# [7] RAG 파이프라인 로드  (매 세션 시작 시 실행)
#
# 아래 3가지를 순서대로 초기화:
#   1) nlpai-lab/KURE-v1 임베딩 모델 로드 (1024d, BGE-M3 기반)
#   2) chunks.json 로드 -> BM25 인덱스 + id->idx 딕셔너리 구축
#   3) Qwen/Qwen2.5-3B-Instruct 4-bit NF4 로드
#
# 첫 실행: HuggingFace 다운로드 (약 4GB, 10~20분)
# 이후   : Colab 캐시 사용 (같은 세션 내 재실행 시 수분 이내)
from src.rag.pipeline import RAGPipeline

print('RAGPipeline 로드 중...')
print('(첫 실행 시 모델 다운로드로 10~20분 소요 — 기다리세요)')

pipeline = RAGPipeline()

print('')
print('RAGPipeline 로드 완료')
print('  임베딩 : nlpai-lab/KURE-v1 (1024d, 한국어 retrieval 특화)')
print('  LLM    : Qwen/Qwen2.5-3B-Instruct (4-bit NF4)')
print('  검색   : BM25(Kiwi) + 임베딩(KURE-v1) 하이브리드 RRF k=60')

In [ ]:
# [8] 테스트 쿼리 — 파이프라인 동작 확인
test_queries = [
    '장학금 신청 방법',
    '수강신청은 언제 해야 하나요',
    '기숙사 입주 신청',
    '취업 지원 프로그램',
    '졸업 요건',
]

sep = '=' * 55
for q in test_queries:
    print('')
    print(sep)
    print('질문: ' + q)
    ans = pipeline.generate(q)
    preview = ans[:300] + ('...' if len(ans) > 300 else '')
    print('답변: ' + preview)

In [ ]:
# [9] Gradio UI — 브라우저에서 챗봇 사용
# share=True: 공개 URL 자동 생성 (72시간 유효)
import gradio as gr

def chat(message, history):
    if not message.strip():
        return '질문을 입력해주세요.'
    return pipeline.generate(message)

demo = gr.ChatInterface(
    fn=chat,
    title='충남대학교 Q&A 챗봇',
    description='학사공지, 장학금, 기숙사, 취업 등 충남대 관련 질문을 입력하세요.',
    examples=[
        '장학금 신청은 어떻게 하나요?',
        '수강신청 일정이 언제인가요?',
        '기숙사 입주 신청 방법이 궁금합니다.',
        '취업 지원 프로그램이 있나요?',
    ],
)
demo.launch(share=True)

In [ ]:
# [10] 평가 일괄 처리 — 교수님 제출용
# questions.json 을 Drive 의 프로젝트 루트(PROJECT)에 업로드 후 실행
# 결과: PROJECT/answers.json 생성
import json, os

questions_path = os.path.join(PROJECT, 'questions.json')
answers_path   = os.path.join(PROJECT, 'answers.json')

if not os.path.exists(questions_path):
    print('questions.json 없음: ' + questions_path)
    print('교수님 제공 파일을 ' + PROJECT + ' 에 업로드 후 재실행')
else:
    with open(questions_path, encoding='utf-8') as f:
        data = json.load(f)

    # 입력 형식 자동 감지: 문자열 리스트 또는 딕셔너리 리스트
    questions = [d['question'] if isinstance(d, dict) else d for d in data]
    print('질문 ' + str(len(questions)) + '건 처리 시작...')

    results = []
    for i, q in enumerate(questions):
        print('[' + str(i + 1) + '/' + str(len(questions)) + '] ' + q[:50])
        answer = pipeline.generate(q)
        results.append({'question': q, 'answer': answer})

    with open(answers_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print('')
    print('완료: ' + answers_path)
    print('총 ' + str(len(results)) + '건 — Drive 에서 answers.json 다운로드하세요')